<div align="center">

# Sherlock, the Robot Detective
A crime solving Production Engine

Group Assignment - Knowledge Representation & Reasoning

Fall, 2026
</div>

![](images/Sherlock.png)

Introducing our production engine, Sherlock, the Robot Detective, here investigating his own footprints.

# Task 1 - A monotonic production system murder mystery

At Ravenwick Manor, Lord Ravenwick has been found
murdered in his study with a letter opener. A threatening letter is in the
library and a forensic sample vial is in the laboratory. **Clara** (the niece),
**Victor** (the business partner), and **Iris** (the assistant) all had motives.

Sherlock must use the available evidence to determine which suspect is most
strongly implicated by the rules, using forward and backward chaining.

In [8]:
from reasoning.engine import WorkingMemory, forward_chain, backward_answers
from reasoning.domains import TASK1_INITIAL_FACTS, TASK1_RULES, TASK1_EXPECTED_KILLER

### The facts Sherlock knows at the start:

In [9]:
wm = WorkingMemory(TASK1_INITIAL_FACTS)
print(f"Initial facts: {len(wm.facts)}")
for fact in sorted(wm.facts):
    print(fact)

Initial facts: 30
('author_of', 'clara', 'letter')
('crime_scene', 'study')
('dismissed_by', 'iris', 'lord_ravenwick')
('fingerprints_on', 'clara', 'letter_opener')
('fingerprints_on', 'victor', 'garden_key')
('found_in', 'letter', 'library')
('found_in', 'letter_opener', 'study')
('found_in', 'sample_vial', 'laboratory')
('inherits_from', 'clara', 'lord_ravenwick')
('linked_clue', 'letter', 'sample_vial')
('linked_clue', 'sample_vial', 'letter_opener')
('owes_money_to', 'victor', 'lord_ravenwick')
('present_during_murder', 'clara')
('present_during_murder', 'iris')
('present_during_murder', 'victor')
('room', 'laboratory')
('room', 'library')
('room', 'study')
('sample_identifies', 'sample_vial', 'clara')
('sample_matches', 'sample_vial', 'letter_opener')
('seen_near', 'clara', 'study')
('seen_near', 'iris', 'library')
('seen_near', 'victor', 'study')
('suspect', 'clara')
('suspect', 'iris')
('suspect', 'victor')
('threatens', 'letter', 'lord_ravenwick')
('victim', 'lord_ravenwick')
(

### The rules:

In [10]:
print(f"Rules: {len(TASK1_RULES)}")
for i, rule in enumerate(TASK1_RULES, 1):
    print(f"{i:2}. IF {rule['if']} THEN {rule['then']}")

Rules: 9
 1. IF [('inherits_from', '?person', 'lord_ravenwick')] THEN ('has_motive', '?person')
 2. IF [('owes_money_to', '?person', 'lord_ravenwick')] THEN ('has_motive', '?person')
 3. IF [('dismissed_by', '?person', 'lord_ravenwick')] THEN ('has_motive', '?person')
 4. IF [('seen_near', '?person', 'study'), ('present_during_murder', '?person')] THEN ('has_opportunity', '?person')
 5. IF [('fingerprints_on', '?person', '?object'), ('weapon', '?object'), ('sample_matches', '?sample', '?object'), ('sample_identifies', '?sample', '?person')] THEN ('physical_evidence', '?person')
 6. IF [('author_of', '?person', '?document'), ('threatens', '?document', 'lord_ravenwick')] THEN ('threatening_message', '?person')
 7. IF [('suspect', '?person'), ('has_motive', '?person'), ('has_opportunity', '?person'), ('physical_evidence', '?person'), ('threatening_message', '?person')] THEN ('identified_killer', '?person')
 8. IF [('linked_clue', '?start', '?end')] THEN ('clue_path', '?start', '?end')
 9.

### Sherlock gathers all the information by starting with the evidence he has and applies the rules repeatedly until he discovers no new fact:

In [11]:
wm_forward = WorkingMemory(TASK1_INITIAL_FACTS)
forward_chain(wm_forward, TASK1_RULES)
new_facts = wm_forward.facts - TASK1_INITIAL_FACTS
print(f"Derived {len(new_facts)} new facts:")
for f in sorted(new_facts):
    print(f)
print("\nResult:", sorted(f for f in wm_forward.facts if f[0] == "identified_killer"))

Derived 11 new facts:
('clue_path', 'letter', 'letter_opener')
('clue_path', 'letter', 'sample_vial')
('clue_path', 'sample_vial', 'letter_opener')
('has_motive', 'clara')
('has_motive', 'iris')
('has_motive', 'victor')
('has_opportunity', 'clara')
('has_opportunity', 'victor')
('identified_killer', 'clara')
('physical_evidence', 'clara')
('threatening_message', 'clara')

Result: [('identified_killer', 'clara')]


![](images/EvidenceBoard.png)

![](images/EvidenceBoard.png)

### Sherlock needs to be sure that the identified killer is really the killer so he starts from the goal "Is Clara the killer?"

In [12]:
wm_backward = WorkingMemory(TASK1_INITIAL_FACTS)
for person in ("clara", "victor", "iris"):
    answers = backward_answers(wm_backward, ("identified_killer", person), TASK1_RULES)
    print(f"Can rules identify {person}?", bool(answers))
print("Who is identified?", backward_answers(wm_backward, ("identified_killer", "?who"), TASK1_RULES))

Can rules identify clara? True
Can rules identify victor? False
Can rules identify iris? False
Who is identified? {('identified_killer', 'clara')}


### Sherlock tries to connect the different pieces of evidence to see how they are related. He uses the rules to find a connection between the threatening letter and the letter opener, and checks if both forward and backward chaining can prove this connection:

In [13]:
path = ("clue_path", "letter", "letter_opener")
print("Forward derives path:", path in wm_forward.facts)
print("Backward proves path:", bool(backward_answers(WorkingMemory(TASK1_INITIAL_FACTS), path, TASK1_RULES)))

Forward derives path: True
Backward proves path: True


### Finally, Sherlock checks if the reasoning system works as expected. He makes sure that the original facts are still there, that both forward and backward chaining identify the same killer, and that backward chaining does not change the original evidence:

In [14]:
assert TASK1_INITIAL_FACTS <= wm_forward.facts
assert {f for f in wm_forward.facts if f[0] == "identified_killer"} == {TASK1_EXPECTED_KILLER}
assert backward_answers(WorkingMemory(TASK1_INITIAL_FACTS), ("identified_killer", "?who"), TASK1_RULES) == {TASK1_EXPECTED_KILLER}
assert wm_backward.facts == TASK1_INITIAL_FACTS
print("All Task 1 checks passed!")

All Task 1 checks passed!
